In [3]:
#Extraction de contenu d’un fichier PDF
#Chargement et extraction de contenu d’un fichier PDF avec LangChain (PyPDFLoader)
from langchain_community.document_loaders import PyPDFLoader

In [4]:
loader = PyPDFLoader("acmecorp-employee-handbook.pdf")
data = loader.load()
print(data)

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-11-20T23:23:16+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2025-11-20T23:23:16+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'acmecorp-employee-handbook.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Employee Handbook\nNon-Disclosure Agreement (NDA) Policy\nEmployees must protect confidential information belonging to the company, its clients, and partners.\nThis includes, but is not limited to, product roadmaps, customer data, internal communications,\nproprietary algorithms, financial information, and unreleased features. Confidential information may not\nbe shared with unauthorized individuals inside or outside the organization. These obligations continue\nafter employment ends.\nWorkplace Conduct Policy\nEmployees must maintain a respectful, professional environment free from

In [5]:
#Segmentation de texte
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(data)
print(len(all_splits))

3


In [6]:
#Transformation des chunks en vecteurs sémantiques
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings( model_name="sentence-transformers/all-MiniLM-L6-v2"
)

C:\Users\dal\AppData\Local\Temp\ipykernel_11240\384325791.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings( model_name="sentence-transformers/all-MiniLM-L6-v2"
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1196.51it/s]


In [7]:
#Création d’une base vectorielle
from langchain_core.vectorstores import InMemoryVectorStore
vector_store = InMemoryVectorStore(embeddings)

In [8]:
#Indexation des documents
ids = vector_store.add_documents(documents=all_splits)

In [9]:
#Recherche sémantique dans une base vectorielle
results = vector_store.similarity_search(
"How many days of vacation does an employee get in their first year?"
)
print(results[0])

page_content='prohibited. Violations may result in disciplinary action.
Paid Time Off (PTO) Policy
Full■time employees accrue PTO according to the following schedule:  0–1 years of service: 10 days
per year (0.833 days per month)  1–3 years of service: 15 days per year (1.25 days per month)  3+
years of service: 20 days per year (1.67 days per month) PTO may be used for vacation, personal
needs, or illness. Requests should be submitted in advance through the HR system unless related to
an emergency. Employees may carry over up to 5 unused PTO days per calendar year. Extended
absences exceeding 5 consecutive business days require manager approval.
Travel & Expense Policy
Employees may be reimbursed for reasonable and necessary expenses incurred during approved
business travel. This includes transportation, lodging, meals, and incidental expenses within
established limits. Receipts must be submitted within 14 days of travel. First-class travel, personal' metadata={'producer': 'ReportL

In [12]:
#AGENT RAG
from langchain_core.tools import tool
@tool
def search_handbook(query: str) -> str:
    """Search the employee handbook for relevant information."""
    results = vector_store.similarity_search(query)
    return results[0].page_content

In [13]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage


from langchain_ollama import ChatOllama
# Initialiser le modèle Ollama
model = ChatOllama(
model="llama3.2:3b", # ou mistral, gemma, etc.
temperature=0
)
agent = create_agent(
        model=model,
        tools=[search_handbook],

        system_prompt="You are a helpful agent that can search the employee hand-book for information.")
response = agent.invoke(

        {"messages": [HumanMessage(content="How many days of vacation does an em-ployee get in their first year?")]})
print(response['messages'][-1].content)

An employee in their first year is entitled to 10 days of vacation per year, which translates to approximately 0.833 days per month.
